# Fraaie plaatjes

## Voorbereiding

Download eerst {download}`Python plaatjes <../problems/assets/fraaie_plaatjes.zip>`.

Dit bestand moet ergens uitgepakt worden. Het bevat een aantal bestanden die allemaal in **dezelfde** directory moeten staan:

- `fraaie_plaatjes.py` (het bestand dat je gaat uitvoeren!)
- `spam.png`
- `in.png`
- `out.png`
- `zp7.png`
- `png.py`

Het bestand dat je gaat bewerken is de afbeelding `spam.png`.  Verder zijn er nog twee andere voorbeelden:

- het meegeleverde bestand `in.png`: ![in.png](images/6/in.png)
- het aangepaste (geïnverteerde) bestand `out.png`: ![out.png](images/6/out.png)

Hier is ook de originele en een geïnverteerde afbeelding (een negatief) van een bekend gebouw op de Zernikecampus in Groningen, de Van Olsttoren:

![Van Olsttoren](images/6/zp7.png)

![Van Olsttoren, geïnverteerd](images/6/invert.png)


````{note}
Naast het importeren van bestaande (ingebouwde) Python-modules kan je ook uit eigen Python bestanden importeren. Je kan dan eigen functies gebruiken uit een ander Python-bestand en dit is een handige techniek voor het hergebruiken van code of een groter programma op te splitsen in verschillende delen.

Bovenaan `fraaie_plaatjes.py` staat de volgende code

```python
from png import *
```

Deze aanroep gebruikt het bestand `png.py` dat ook in de directory staat die je zojuist hebt gedownload. Dit Python-bestand bevat functionaliteit voor het openen, lezen en schrijven van afbeeldingen. Hier gebruiken we `*` om *alle* functies in het bestand `png.py` te importeren en te kunnen gebruiken.
````

## Onze PNG module

Voor dit probleem ga je gebruik maken van een Python module die `png`-afbeeldingen leest en schrijft. Gelukkig hebben zowel macOS, Linux als Windows ingebouwde programma's (Preview, EoG, Paint) die bijna elke afbeelding omzetten naar het "portable network graphics" formaat (oftewel PNG).


````{important}
De meegeleverde `png.py` is de facade voor het lezen en schrijven. Je hoeft `png.py` of Pillow niet zelf te ontwerpen. Gebruik in je programma alleen deze drie functies:

- `get_rgb(filename)` leest een PNG-bestand en geeft een afbeelding terug.
- `save_rgb(pixels, filename)` schrijft de afbeelding `pixels` naar een PNG-bestand.
- `get_wh(pixels)` geeft `(width, height)` terug: eerst de breedte, dan de hoogte.

Een afbeelding is een lijst van rijen, een rij is een lijst van pixels en een pixel is een lijst met drie waarden `[R, G, B]`. De facade kan intern een pixel als tuple aanleveren; lees hem dan met indexen, bijvoorbeeld `pixel[0]`, en bouw voor je nieuwe afbeelding steeds een nieuwe pixel en nieuwe rij als lijst.
````

## Opdracht 1: Uitproberen

Probeer `fraaie_plaatjes.py` op de gebruikelijke manier uit. 

Door dit te doen wordt de functie `invert()` uitgevoerd (kijk naar het einde van `fraaie_plaatjes.py`, je ziet de aanroep van `invert` bijna aan het einde van het bestand...).

Voor de volledigheid is hier de code van deze functie:

```python
def invert():
    """Voer deze functie uit om de afbeelding in.png te lezen,
    aan te passen en het resultaat weg te schrijven naar out.png
    """
    im_pix = get_rgb("in.png")  # lees het bestand in.png in

    print("De eerste twee pixels van de eerste rij zijn", im_pix[0][0:2])

    # Onthoud dat im_pix een lijst (de afbeelding) van
    # lijsten (elke rij) van lijsten (elke pixel is [R,G,B]) is

    new_pix = []

    for row in im_pix:
        new_row = []

        for pixel in row:
            new_pixel = change(pixel)
            new_row += [new_pixel]

        new_pix += [new_row]

    # sla nu het bestand 'out.png' op
    save_rgb(new_pix, "out.png")


# uitproberen!
invert()
```

Lees deze functie door om een idee te krijgen van hoe je afbeeldingen kan inlezen, wijzigen en uitvoeren. Denk alvast na over de datastructuur van  `im_pix`, en de andere met dezelfde structuur, `new_pix`. Het volgende zal je hier bij helpen...

### Datastructuur

De relevante datastructuur in bovenstaande code is `im_pix`, die alle pixelgegevens van de afbeelding bevat.

Maar *welke vorm* heeft deze datastructuur? Het is noodzakelijk dit te weten om de data aan te kunnen passen!

`im_pix` is een lijst van *rijen pixels*.

* Elke *rij pixels* bestaat op zijn beurt uit een lijst van *pixels*.
* Elke *pixel* bestaat op zijn beurt uit een lijst van drie integers: de rood-, groen- en blauwwaarde van de pixel (elk van 0 tot en met 255).

Hier is een voorbeeld van een volledige afbeelding van 2x3 pixels (met twee rijen van drie pixels per rij):

```python
im_pix = [
    [[0, 0, 255], [0, 0, 255], [0, 0, 0]],  # eerste rij van drie pixels
    [[255, 255, 255], [255, 0, 0], [0, 0, 255]],  # tweede rij van drie pixels
]
```

In dit voorbeeld zijn de pixels linksboven en middenboven puur blauw, en de pixel rechtsboven is zwart. De pixel linksonder is wit, de pixel middenonder is rood en de pixel rechtsonder is blauw.

Deze pixels zien er in kleur als volgt uit:

![Voorbeeld](images/6/pix.png)

De geneste lus in de functie `invert` heeft twee niveaus:

-   Het buitenste niveau loopt door elke rij van de afbeelding (elke rij wordt `row` genoemd, de variabele voor dat gedeelte)
    -   De code voor dit buitenste niveau is `for row in im_pix`

-   Het binnenste niveau loopt door elke pixel van de afbeelding heen (elke pixel krijgt de naam `pixel`, de variabele voor dit gedeelte)
    -   De code voor dit binnenste niveau is `for pixel in row`

-   Vergeet niet dat `pixel` zelf een lijst van drie integers is: `[red, green, blue]`

Met dit model in gedachten ben je klaar om de gegevens te "herschikken" zoals je dat wilt! Dat is de volgende stap...

### Luminantie

Eerst zullen we ons richten op operaties op pixelniveau. Voor het maken van *grijswaarden*- en *binaire* beelden zal je je moeten richten op de [relatieve luminantie](http://en.wikipedia.org/wiki/Luminance_(relative)). In essentie is dit hoe helder of donker de kleuren in een pixel zijn (in vergelijking met wit).

Zoals [Wikipedia](http://en.wikipedia.org/wiki/Luminance_(relative)) het berekent, is de luminantie 21% rood, 72% groen en 7% blauw. Intuïtief is dit logisch, want als je denkt aan standaard rood, groen en blauw, dan is groen het lichtste en heeft dus het grootste positieve effect op de luminantie, terwijl blauw donkerder is en een lagere waarde heeft voor de luminantie. Dit is nuttig! Je gaat de luminantie berekenen voor pixelbewerkingen.

### Spelen met pixels

We hebben je een functie `invert` gegeven en die een beeld wijzigt om het negatief te creëren. Dat wil zeggen, alle kleurwaarden zijn 255 min hun oorspronkelijke waarde. Let vooral op het gebruik van de lus in `invert` (`for pixel in row`), die elke pixel in de afbeelding langsloopt (itereert) en `change(p)` aanroept.

````{attention}
Je gaat straks een aantal nieuwe functies schrijven en het is het gemakkelijkst om bij deze functies dezelfde structuur te gebruiken zoals in `invert`: een geneste lus waarin je voor elke pixel een hulpfunctie aanroept (de functie `change` in het geval van `invert`).
````

## Opdracht 2: `greyscale()`

Schrijf nu een functie `greyscale` die een afbeelding wijzigt naar grijswaarden. Hiervoor zul je iets moeten doen dat lijkt op `invert`, behalve dat de nieuwe functie `change` de luminantie van de pixel zal berekenen zoals hierboven beschreven. Aangezien luminantie een maat is van hoe wit of zwart een pixel is, is het eenvoudig om een lijst met RGB-waarden in grijswaarden terug te geven, zie onze eerdere opmerkingen over luminantie en hoe het een bepaald percentage is voor elke kleur!

![Spam](images/6/spam.png)

![Grijswaarden](images/6/greyscale.png)

*Een afbeelding voor en na de conversie naar grijswaarden (luminantie).*

Krijg je de fout `TypeError: 'float' object cannot be interpreted as an integer`? Dat komt doordat je luminantieberekening een kommagetal oplevert, terwijl `save_rgb` gehele RGB-waarden nodig heeft. Gebruik `int()` om de uitkomst om te zetten.

Als je de fout `filename is not defined` krijgt, controleer of er aanhalingstekens om de bestandsnaam staan en dat het bestand aanwezig is in dezelfde directory als je Python script.

Krijg je de fout `filename is not defined`? Zorg ervoor dat er  aanhalingstekens om de bestandsnaam staan wanneer je deze aanroept in de IPython shell. Zorg er ook voor dat de afbeelding in dezelfde directory staat als je Python-script, en dat de afbeelding de juiste permissies heeft.

## Opdracht 3: `binarize(thresh)`

Schrijf een functie `binarize(thresh)`, die een afbeelding binair (zwart-wit) maakt met een drempelwaarde `thresh` gegeven door de gebruiker. Deze drempelwaarde is een helderheidswaarde tussen 0 en 255, als een pixel groter is dan de drempelwaarde, dan moet het wit worden, en als het minder is dan de drempelwaarde, dan moet het zwart worden. Dus, een drempelwaarde van 0 betekent dat je afbeelding zuiver wit wordt en een drempelwaarde van 255 betekent dat je afbeelding zwart wordt.

![Binaire afbeelding](images/6/binarize.png)

*Binaire spam met een grenswaarde van 100.*

## Geometrische transformaties

### Opdracht 4: `flip_vert()`

Schrijf de functie `flip_vert`, waarbij de afbeelding op de *horizontale* as wordt omgedraaid (de onderkant is aan de bovenkant en de bovenkant aan de onderkant). Je gebruikt hiervoor dezelfde basisstructuur als bij de eerdere opgaven, een hoofdfunctie die het bestand opent en met een geneste lus de hulpfunctie voor elke pixel aanroept.

 In `flip_vert` zal je alleen de rijen moeten langslopen in plaats van de pixels in `im_pix`, en de volgorde omkeren. Onthoud dat als `lst` een lijst is, dan is `lst[::-1]` het omgekeerde van die lijst.

![Verticaal omgedraaid](images/6/flip_vert.png)

*`in.png` verticaal omgedraaid.*

### Opdracht 5: `flip_horiz()`

Schrijf de functie `flip_horiz`, die een afbeelding draai om zijn *verticale* as. Dit moet op dezelfde manier werken als `flip_vert`, maar draait in de horizontale richting. In plaats van de rijen te verschuiven zal je nu moeten bedenken hoe de pixels in de rijen worden verschoven als een afbeelding horizontaal wordt gespiegeld. Merk op, `in.png` horizontaal spiegelen heeft geen effect omdat de afbeelding symmetrisch is ten opzichte van de verticale as...

![Horizontaal omgedraaid](images/6/flip_horiz.png)

*MAPS!*

### Opdracht 6: `mirror_vert()`

Schrijf de functie `mirror_vert`, die de afbeelding over de horizontale as (d.w.z. dat het bovenste deel ondersteboven wordt gespiegeld aan de onderkant van het beeld). Het eenvoudigst is om de onderste helft van `im_pix` te vervangen door de omgekeerde rijen van de bovenste helft. Gebruik `get_wh(im_pix)` om de afmetingen te krijgen; het resultaat is `(width, height)`, dus de tweede waarde is de hoogte.

![Verticaal gespiegeld](images/6/mirror_vert.png)

*`in.png` verticaal gespiegeld*

```{warning}
Maak voor een bewerking steeds nieuwe lijsten. `list1 = list2` geeft alleen een tweede naam voor dezelfde lijst; verander de invoer daarom niet tijdens een transformatie. Je hoeft dit gedrag nog geen naam te geven.
```

### Opdracht 7: `mirror_horiz()`

Schrijf de functie `mirror_horiz` die gelijk is aan de functie `mirror_vert`, maar dan over de verticale as. In plaats van de onderste rijen te vervangen door de omgekeerde bovenste rijen (zoals in `mirror_vert`), vervang je de laatste helft van de pixels in elke rij door de omgekeerde eerste helft van de pixels.

![Horizontaal gespiegeld](images/6/mirror_horiz.png)

*De belangrijkere vraag is, welke kant maak je eerst open?*

### Opdracht 8: `scale()`

Schrijf de functie `scale`, die de afbeelding verkleint naar de helft van de oorspronkelijke afmetingen (dit zal een kwart van het oorspronkelijke oppervlakte zijn). De eenvoudigste manier om dit te doen is om elke tweede pixel in elke rij te verwijderen (het afbeelding horizontaal te schalen) *en* elke tweede rij te verwijderen (de afbeelding verticaal te schalen).

### Opdracht 9 Meer transformaties

Mocht je jouw eigen effecten willen creëren, dan willen we ze graag zien! Voeg zeker een commentaar of toe om uit te leggen wat je hebt gedaan. Laat je fantasie de vrije loop!

Voel je ook vrij om een paar afbeeldingen naar eigen keuze toe te voegen die je algoritmisch hebt aangepast...